In [19]:
# packages
import pandas as pd
from mod02_build_bot_predictor import train_model

### Define a function to extract predictions from the model

In [20]:
def predict_bot(df, model=None):
    """
    Predict whether each account is a bot (1) or human (0).
    """
    if model is None:
        model = train_model()

    preds = model.predict(df)
    return pd.Series(preds, index=df.index)

### Define a function to evaluate model error

In [21]:
def confusion_matrix_and_metrics(y_true, y_pred):
    """
    Computes confusion matrix and common error rates for binary classification.

    Assumes labels:
      0 = negative class
      1 = positive class

    Returns:
      dict with:
        tn, fp, fn, tp
        misclassification_rate
        false_positive_rate
        false_negative_rate
    """
    tn = fp = fn = tp = 0

    for yt, yp in zip(y_true, y_pred):
        if yt == 0 and yp == 0:
            tn += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 1 and yp == 0:
            fn += 1
        elif yt == 1 and yp == 1:
            tp += 1
        else:
            raise ValueError("Labels must be 0 or 1")

    total = tn + fp + fn + tp

    misclassification_rate = (fp + fn) / total if total > 0 else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "misclassification_rate": misclassification_rate,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
    }


### Load the data

In [22]:
TRAIN_PATH = "mod02_data/train.csv"
train = pd.read_csv(TRAIN_PATH)

TEST_PATH = "mod02_data/test.csv"
test = pd.read_csv(TEST_PATH)

### Format the data by independent vs. dependent variables

In [23]:
X_train = train.drop(columns=["is_bot"])
y_train = train['is_bot']

X_test = test.drop(columns=["is_bot"])
y_test = test['is_bot']

### Build the model on training data

In [24]:
model = train_model(X_train, y_train)

### Get the model predictions on training and test data

In [25]:
y_pred_train = predict_bot(X_train, model)
y_pred_test = predict_bot(X_test, model)

### Check results on the training set (data used to build the model)

In [26]:
confusion_matrix_and_metrics(y_train, y_pred_train)

{'tp': 157,
 'tn': 2626,
 'fp': 11,
 'fn': 206,
 'misclassification_rate': 0.07233333333333333,
 'false_positive_rate': 0.004171406901782328,
 'false_negative_rate': 0.5674931129476584}

### Check results on the test set (new data not yet seen by the model)

In [27]:
confusion_matrix_and_metrics(y_test, y_pred_test)

{'tp': 31,
 'tn': 855,
 'fp': 19,
 'fn': 95,
 'misclassification_rate': 0.114,
 'false_positive_rate': 0.021739130434782608,
 'false_negative_rate': 0.753968253968254}

# Discussion Questions

### Based on the misclassification rate of your model, discuss your confidence in the ability to predict a bot. 

With the misclassification rate being roughly 11%-12% I am fairly confident that my model could correctly identify most of the cases. However, I would not say it is reliable in real world scenarios, or at least scenarios where precision is necessary. Maybe in situations where a broad estimate can help metrics, but nothing more.

### What are potential ramifications of false positives from the model?

The most obvious ramification is that a human could be incorrectly identified as a bot, which, depending on the following actions from the software, could be bad. Another ramification is the classification of specific human trends or data yielded by the model would be labeled as representatitive of a bot, which, depending on the usage of that data, could be problematic.

### What are potential ramifications of false negatives from the model?

The most dangerous ramification is that bots incorrectly identified as humans would remain in the clear. Depending on the purpose of the said bot, this could be negative for whatever platform they reside on. This also relates to the concerns of the previous question (but revered), where certain trends identified as "human" by the model would actually be "bot" tendencies, which would allow the bots and creators of said bots to adapt to fit those tendencies that slip by the model.